# Annotation Converter Tutorial

This notebook demonstrates all **6 pairwise conversions** supported by `AnnotationConverter` between three annotation formats used in the bird detection project:

| # | Source Format | Destination Format | Method |
|---|---------------|--------------------|--------|
| 1 | CVAT XML | YOLO `.txt` | `cvat_to_yolo()` |
| 2 | CVAT XML | COCO JSON | `cvat_to_coco()` |
| 3 | COCO JSON | YOLO `.txt` | `coco_to_yolo()` |
| 4 | COCO JSON | CVAT XML | `coco_to_cvat()` |
| 5 | YOLO `.txt` | COCO JSON | `yolo_to_coco()` |
| 6 | YOLO `.txt` | CVAT XML | `yolo_to_cvat()` |

**Bonus:** `create_yaml_config()` generates a YOLO dataset YAML from the current class mapping.

All outputs are written to `annotations/conversion_tutorial_outputs/`.

In [ ]:
import sys
import json
from pathlib import Path

# Notebook lives in utilities/; repo root is one level up
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root / 'utilities'))

from annotation_converter import AnnotationConverter

# ── Input paths ──────────────────────────────────────────────────────────────
CVAT_XML    = repo_root / 'annotations/annotations.xml'
COCO_JSON   = repo_root / 'annotations/H03.json'
YOLO_LABELS = repo_root / 'data/h03/labels/train'

# ── Output base ──────────────────────────────────────────────────────────────
OUT = repo_root / 'annotations/conversion_tutorial_outputs'
OUT.mkdir(parents=True, exist_ok=True)

print('repo_root :', repo_root)
print('CVAT_XML  :', CVAT_XML.exists(),  CVAT_XML)
print('COCO_JSON :', COCO_JSON.exists(), COCO_JSON)
print('YOLO_LABELS:', YOLO_LABELS.exists(), YOLO_LABELS)
print('OUT       :', OUT)

## Input Formats Overview

### CVAT XML (track-based)
Produced by the CVAT annotation tool for video sequences. Bounding boxes live inside `<track>` elements that span multiple frames. Each `<box>` carries `frame`, `xtl`, `ytl`, `xbr`, `ybr` attributes in absolute pixels. Image dimensions come from `<task>/<original_size>` metadata.

```xml
<annotations>
  <task><id>1</id><original_size><width>1280</width><height>720</height></original_size></task>
  <track id="0" label="Bird" task_id="1">
    <box frame="0" xtl="120.5" ytl="88.0" xbr="145.2" ybr="110.3" outside="0"/>
    ...
  </track>
</annotations>
```

### COCO JSON
Standard COCO format with three top-level keys: `images`, `annotations`, `categories`. Bounding boxes use `[x, y, width, height]` in absolute pixels (top-left origin).

```json
{
  "images":      [{"id": 1, "file_name": "frame_000000.png", "width": 1280, "height": 720}],
  "annotations": [{"id": 1, "image_id": 1, "category_id": 1, "bbox": [120.5, 88.0, 24.7, 22.3]}],
  "categories":  [{"id": 1, "name": "bird"}]
}
```

### YOLO `.txt`
One text file per image, one line per object: `<class_id> <x_center> <y_center> <width> <height>` all normalized to `[0, 1]` by image dimensions. Class IDs are 0-indexed integers.

```
0 0.103516 0.137500 0.019531 0.031250
0 0.421875 0.562500 0.025000 0.041667
```

In [ ]:
import xml.etree.ElementTree as ET

# ── CVAT XML: show first track ────────────────────────────────────────────────
print('=== CVAT XML: first track ===')
tree = ET.parse(CVAT_XML)
root = tree.getroot()
first_track = root.find('.//track')
print(f'  label     : {first_track.get("label")}')
print(f'  task_id   : {first_track.get("task_id")}')
boxes = first_track.findall('box')
print(f'  box count : {len(boxes)}')
b = boxes[0]
print(f'  first box : frame={b.get("frame")} xtl={b.get("xtl")} ytl={b.get("ytl")} '
      f'xbr={b.get("xbr")} ybr={b.get("ybr")} outside={b.get("outside")}')

# ── COCO JSON: show first 2 images + first 2 annotations ─────────────────────
print('\n=== COCO JSON: summary ===')
with open(COCO_JSON) as f:
    coco = json.load(f)
print(f'  images      : {len(coco["images"]):,}')
print(f'  annotations : {len(coco["annotations"]):,}')
print(f'  categories  : {coco["categories"]}')
print(f'  images[0]   : {coco["images"][0]}')
print(f'  images[1]   : {coco["images"][1]}')
print(f'  annotations[0]: {coco["annotations"][0]}')
print(f'  annotations[1]: {coco["annotations"][1]}')

# ── YOLO labels: show a sample .txt file ─────────────────────────────────────
print('\n=== YOLO label sample ===')
sample_txts = sorted(YOLO_LABELS.glob('*.txt'))[:5]
print(f'  Total .txt files: {len(list(YOLO_LABELS.glob("*.txt"))):,}')
print(f'  First label file: {sample_txts[0].name}')
with open(sample_txts[0]) as f:
    lines = f.readlines()
print(f'  Lines in file: {len(lines)}')
for line in lines[:5]:
    print(f'  {line.rstrip()}')

---
## Section 1: CVAT XML → YOLO

`cvat_to_yolo()` reads the track-based XML, groups boxes by frame number, and writes one `.txt` label file per frame.  
- Class labels (e.g. `"Bird"`) are mapped to integer IDs via `class_mapping`.
- Absolute pixel coords `(xtl, ytl, xbr, ybr)` are converted to normalized center format.
- `image_name_pattern` controls the output filename stem (default `frame_{:06d}`).

In [ ]:
converter = AnnotationConverter(class_mapping={'Bird': 0})
class_map = converter.cvat_to_yolo(
    cvat_xml_path=CVAT_XML,
    output_dir=OUT / 'cvat_to_yolo',
    image_name_pattern='frame_{:06d}'
)
print('Class mapping:', class_map)

In [ ]:
# Inspect CVAT→YOLO output
yolo_out = OUT / 'cvat_to_yolo'
txt_files = sorted(yolo_out.glob('*.txt'))
print(f'Output .txt files: {len(txt_files)}')
print(f'First few: {[f.name for f in txt_files[:5]]}')

# Show content of first label file
print(f'\nContent of {txt_files[0].name}:')
with open(txt_files[0]) as f:
    for line in f:
        print(' ', line.rstrip())

---
## Section 2: CVAT XML → COCO JSON

`cvat_to_coco()` converts the same track-based XML into COCO JSON.  
- Each annotated frame becomes an entry in `images`.
- `image_name_pattern` sets the `file_name` field (include the extension here).
- Bounding boxes are stored as `[x, y, w, h]` in absolute pixels (COCO convention).

In [ ]:
converter = AnnotationConverter(class_mapping={'Bird': 0})
converter.cvat_to_coco(
    cvat_xml_path=CVAT_XML,
    output_path=OUT / 'cvat_to_coco/annotations.json',
    image_name_pattern='frame_{:06d}.png'
)

In [ ]:
# Inspect CVAT→COCO output
with open(OUT / 'cvat_to_coco/annotations.json') as f:
    out_coco = json.load(f)

print(f'images      : {len(out_coco["images"]):,}')
print(f'annotations : {len(out_coco["annotations"]):,}')
print(f'categories  : {out_coco["categories"]}')
print(f'\nimages[0]      : {out_coco["images"][0]}')
print(f'annotations[0] : {out_coco["annotations"][0]}')

---
## Section 3: COCO JSON → YOLO

`coco_to_yolo()` converts COCO JSON annotations to YOLO `.txt` files.  
- With `use_filename=True` (default), the label file stem is taken from each image's `file_name` (extension stripped).
- Only images that have at least one annotation produce a label file — here **2,631** files from the 27,618-image `H03.json`.
- COCO bbox `[x, y, w, h]` is converted to normalized center format.

In [ ]:
converter = AnnotationConverter(class_mapping={'bird': 0})
converter.coco_to_yolo(
    coco_json_path=COCO_JSON,
    output_dir=OUT / 'coco_to_yolo',
    use_filename=True
)

In [ ]:
# Inspect COCO→YOLO output
coco_yolo_out = OUT / 'coco_to_yolo'
coco_yolo_files = sorted(coco_yolo_out.glob('*.txt'))
print(f'Output .txt files: {len(coco_yolo_files):,}')

# Show content of first 3 label files
for label_file in coco_yolo_files[:3]:
    with open(label_file) as f:
        lines = f.readlines()
    print(f'\n{label_file.name} ({len(lines)} line(s)):')
    for line in lines:
        print(' ', line.rstrip())

---
## Section 4: COCO JSON → CVAT XML

`coco_to_cvat()` produces a CVAT XML file with one `<image>` element per entry in `coco["images"]`.  
- All 27,618 images appear in the XML; unannotated frames have empty `<image>` elements (no `<box>` children).
- Bounding boxes are stored as `xtl/ytl/xbr/ybr` attributes in absolute pixels.

In [ ]:
converter = AnnotationConverter()
converter.coco_to_cvat(
    coco_json_path=COCO_JSON,
    output_path=OUT / 'coco_to_cvat/annotations.xml'
)

In [ ]:
# Inspect COCO→CVAT output: find first <image> with at least one <box>
tree = ET.parse(OUT / 'coco_to_cvat/annotations.xml')
root_out = tree.getroot()

all_images = root_out.findall('image')
print(f'Total <image> elements: {len(all_images):,}')

annotated = [img for img in all_images if len(img) > 0]
print(f'Annotated <image> elements: {len(annotated):,}')

print('\nFirst annotated <image>:')
first_ann = annotated[0]
print(f'  name={first_ann.get("name")} width={first_ann.get("width")} height={first_ann.get("height")}')
for box in list(first_ann)[:3]:
    print(f'  <box label={box.get("label")} xtl={box.get("xtl")} ytl={box.get("ytl")} '
          f'xbr={box.get("xbr")} ybr={box.get("ybr")}>')

---
## Section 5: YOLO → COCO JSON

`yolo_to_coco()` reads every `.txt` file in the labels directory and builds a COCO JSON.  
- `image_size=(1280, 720)` is applied uniformly (all images are the same resolution).
- `class_names={0: 'bird'}` maps YOLO class IDs to human-readable names used in `categories`.
- Every label file becomes one entry in `images`, even if it has zero annotations.

In [ ]:
converter = AnnotationConverter()
converter.yolo_to_coco(
    labels_dir=YOLO_LABELS,
    output_path=OUT / 'yolo_to_coco/annotations.json',
    image_size=(1280, 720),
    class_names={0: 'bird'}
)

In [ ]:
# Inspect YOLO→COCO output
with open(OUT / 'yolo_to_coco/annotations.json') as f:
    y2c = json.load(f)

print(f'images      : {len(y2c["images"]):,}')
print(f'annotations : {len(y2c["annotations"]):,}')
print(f'categories  : {y2c["categories"]}')
print(f'\nimages[0]      : {y2c["images"][0]}')
print(f'annotations[0] : {y2c["annotations"][0]}')

---
## Section 6: YOLO → CVAT XML

`yolo_to_cvat()` converts YOLO `.txt` files to a CVAT XML (image-based, not track-based).  
- `image_size=(1280, 720)` is used to denormalize the bounding boxes back to absolute pixels.
- `class_names={0: 'bird'}` maps class IDs to label strings written into `<box label=...>`.
- The output XML has one `<image>` per label file with `<box>` children.

In [ ]:
converter = AnnotationConverter()
converter.yolo_to_cvat(
    labels_dir=YOLO_LABELS,
    output_path=OUT / 'yolo_to_cvat/annotations.xml',
    image_size=(1280, 720),
    class_names={0: 'bird'}
)

In [ ]:
# Inspect YOLO→CVAT output
tree = ET.parse(OUT / 'yolo_to_cvat/annotations.xml')
root_yc = tree.getroot()

imgs_yc = root_yc.findall('image')
print(f'Total <image> elements: {len(imgs_yc):,}')

# Find first image that has boxes
first_with_box = next((img for img in imgs_yc if len(img) > 0), None)
if first_with_box is not None:
    print(f'\nFirst <image> with boxes:')
    print(f'  name={first_with_box.get("name")} width={first_with_box.get("width")} height={first_with_box.get("height")}')
    for box in list(first_with_box)[:3]:
        print(f'  <box label={box.get("label")} xtl={box.get("xtl")} ytl={box.get("ytl")} '
              f'xbr={box.get("xbr")} ybr={box.get("ybr")}>')
else:
    print('No annotated images found')

---
## Bonus: Create YOLO Dataset YAML

`create_yaml_config()` writes a `dataset.yaml` compatible with Ultralytics YOLO.  
- `dataset_path` is the dataset root (relative path used at training time).
- `train_dir`, `val_dir`, `test_dir` are relative to `dataset_path`.
- The class names are derived from the converter's current `class_mapping`.

In [ ]:
converter = AnnotationConverter(class_mapping={'bird': 0})
converter.create_yaml_config(
    output_path=OUT / 'yaml_config/dataset.yaml',
    dataset_path='data/h03',
    train_dir='images/train',
    val_dir='images/val',
    test_dir='images/test'
)

In [ ]:
# Show generated YAML
yaml_path = OUT / 'yaml_config/dataset.yaml'
print(yaml_path.read_text())

In [ ]:
# Verify all 7 output subdirectories exist
expected_dirs = [
    'cvat_to_yolo',
    'cvat_to_coco',
    'coco_to_yolo',
    'coco_to_cvat',
    'yolo_to_coco',
    'yolo_to_cvat',
    'yaml_config',
]
print('Output directory check:')
for d in expected_dirs:
    path = OUT / d
    contents = list(path.iterdir()) if path.exists() else []
    status = '✓' if path.exists() else '✗'
    print(f'  {status}  {d}/  ({len(contents)} file(s))')

---
## Summary

All 6 pairwise conversions completed successfully. Reference table:

| # | Source | Destination | Method | Key Parameters |
|---|--------|-------------|--------|----------------|
| 1 | CVAT XML | YOLO `.txt` | `cvat_to_yolo()` | `class_mapping={'Bird': 0}`, `image_name_pattern='frame_{:06d}'` |
| 2 | CVAT XML | COCO JSON | `cvat_to_coco()` | `class_mapping={'Bird': 0}`, `image_name_pattern='frame_{:06d}.png'` |
| 3 | COCO JSON | YOLO `.txt` | `coco_to_yolo()` | `class_mapping={'bird': 0}`, `use_filename=True` |
| 4 | COCO JSON | CVAT XML | `coco_to_cvat()` | *(no class_mapping needed — labels from JSON categories)* |
| 5 | YOLO `.txt` | COCO JSON | `yolo_to_coco()` | `image_size=(1280, 720)`, `class_names={0: 'bird'}` |
| 6 | YOLO `.txt` | CVAT XML | `yolo_to_cvat()` | `image_size=(1280, 720)`, `class_names={0: 'bird'}` |

**Class mapping note:** CVAT XML in this project uses `"Bird"` (capital B) while COCO JSON uses `"bird"` (lowercase). Use `class_mapping={'Bird': 0, 'bird': 0}` when a single converter instance needs to handle both sources.